<a href="https://colab.research.google.com/github/alarcon7a/youtube-tutorial-sources/blob/main/Notebooks/Google%20AI/Gen%20Media/Get_started_Omni_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 Google LLC.

In [ ]:
# @title Licenciado bajo la Licencia Apache, Versión 2.0 (la "Licencia");
# no puedes usar este archivo excepto en cumplimiento con la Licencia.
# Puedes obtener una copia de la Licencia en
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# A menos que lo requiera la ley aplicable o se acuerde por escrito, el software
# distribuido bajo la Licencia se distribuye "TAL CUAL", SIN GARANTÍAS NI CONDICIONES
# DE NINGÚN TIPO, expresas o implícitas.
# Consulta la Licencia para conocer el idioma específico que rige los permisos y
# las limitaciones bajo la Licencia.

# Genera y edita videos con Gemini Omni Flash — Tutorial paso a paso 🎬

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_Omni.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Ejecutar en Google Colab</a>
<a target="_blank" href="https://github.com/google-gemini/cookbook/blob/main/quickstarts/Get_started_Omni.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />Ver en GitHub</a>

**Gemini Omni Flash** es el modelo de generación y edición de video de Google, expuesto a través de la **Interactions API**. En un solo modelo unificado puedes:

- Generar video **desde texto** (text-to-video).
- Animar una **imagen** para convertirla en video (image-to-video).
- Usar **imágenes de referencia** para inyectar personajes, objetos o estilos en una escena nueva.
- Hacer **interpolación de keyframes** (frame inicial + frame final).
- **Extender** videos existentes con continuidad temporal.
- **Editar videos ya existentes** —incluidos videos tuyos, grabados con tu propio celular— con simples instrucciones en lenguaje natural o apoyándote en imágenes de referencia.

En este tutorial vamos a recorrer todas estas capacidades con ejemplos variados (naturaleza, ciencia, publicidad, storytelling con personajes...) y, en la parte final, vas a poder **subir un video tuyo** y editarlo de dos formas distintas: solo con prompt de texto, y combinando prompt + imagen de referencia.

> 💡 Este notebook está basado en el ejemplo oficial `Get_started_Omni.ipynb` del equipo de Gemini, traducido, reordenado como tutorial progresivo y ampliado con ejemplos propios.

## Índice

1. [Configuración inicial](#scrollTo=setup)
2. [Tu primer video: text-to-video](#scrollTo=primer_video)
3. [Entendiendo la respuesta de la API](#scrollTo=entendiendo)
4. [Funciones auxiliares para mostrar videos](#scrollTo=helpers)
5. [Controlando el "pensamiento" del modelo](#scrollTo=thinking)
6. [Relación de aspecto: horizontal y vertical](#scrollTo=aspect_ratio)
7. [Resolución: de borrador rápido a 4K](#scrollTo=resolucion)
8. [Ejemplo creativo: conocimiento del mundo](#scrollTo=world_knowledge)
9. [De imagen a video](#scrollTo=image_to_video)
10. [Video a partir de imágenes de referencia](#scrollTo=reference_to_video)
11. [Interpolación de keyframes](#scrollTo=keyframes)
12. [Control explícito con el parámetro `task`](#scrollTo=task_param)
13. [Escenas con múltiples personajes y referencias](#scrollTo=multi_ref)
14. [Extender un video](#scrollTo=extension)
15. [Edición conversacional multi-turno](#scrollTo=edicion_multiturno)
16. [🎥 Sube TU video y edítalo](#scrollTo=tu_video)
17. [Entrega por URI para producción](#scrollTo=uri_delivery)
18. [Guía de prompting: buenas prácticas](#scrollTo=buenas_practicas)
19. [Veo vs. Omni: ¿cuál elegir?](#scrollTo=veo_vs_omni)
20. [Próximos pasos](#scrollTo=siguientes_pasos)

<a name="setup"></a>
## 1. Configuración inicial

### Instalar el SDK

La Interactions API requiere la versión más reciente del SDK `google-genai`.

In [ ]:
%pip install -U -q "google-genai>=2.10.0"  # 2.10+ es necesario para el parámetro `task`

Para ejecutar las siguientes celdas necesitas tu API key guardada como **Secreto de Colab** con el nombre `GEMINI_API_KEY`. Si aún no tienes una, créala en [Google AI Studio](https://aistudio.google.com/).

In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY =  userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

Selecciona el modelo que quieres usar en esta guía:

In [ ]:
MODEL_ID = "gemini-omni-1.1-flash"  # @param ["gemini-omni-1.1-flash", "gemini-omni-flash-preview"] {"allow-input": true, "isTemplate": true}

<a name="primer_video"></a>
## 2. Tu primer video: text-to-video

Para generar un video puramente a partir de una descripción de texto, llama a `client.interactions.create` con `model=MODEL_ID` y `response_format={"type": "video"}`.

In [ ]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input="Un pinguino corriendo alegremente por un campo de flores silvestres al atardecer, luz cálida y cinematográfica.",
    response_format={"type": "video"},  # Opcional
)

<a name="entendiendo"></a>
## 3. Entendiendo la respuesta de la API

`client.interactions.create(...)` devuelve un objeto `Interaction`. Como Omni es nativamente multimodal y está pensado como una conversación por turnos, la respuesta contiene una secuencia ordenada de **`steps`** que representan la trayectoria de ejecución del modelo.

In [ ]:
import json

# Recortamos los campos 'data' y 'signature' en todos los steps y partes de contenido
print(json.dumps([
    {**s.__dict__,
     **{k: v[:70] + '... (recortado)' for k, v in s.__dict__.items() if k in ['data', 'signature'] and isinstance(v, str) and len(v) > 100},
     'content': [{**c.__dict__,
                  **{k: v[:70] + '... (recortado)' for k, v in c.__dict__.items() if k == 'data' and isinstance(v, str) and len(v) > 100}}
                 for c in getattr(s, 'content', []) or []]}
    for s in interaction.steps
], indent=2, default=str))

### Recorriendo los steps de salida

Cada `step` en `interaction.steps` tiene un `type`:

* `thought`: contiene resúmenes de razonamiento cuando el "thinking" está habilitado.
* `model_output`: contiene el contenido multimedia final generado.

Dentro de un step `model_output`, `step.content` es una lista de partes (`text`, `image`, `audio` o `video`). Puedes recorrerlas manualmente para inspeccionar metadatos o extraer los bytes del video:

In [ ]:
from IPython.display import Video, display

# Recorremos manualmente los steps para encontrar la parte de tipo video
# (usamos getattr porque no todos los tipos de step, p.ej. "thought", exponen 'content')
for step in reversed(interaction.steps):
    if step.type == "model_output" and getattr(step, "content", None):
        for part in reversed(step.content):
            if part.type == "video":
                display(Video(data=part.data, embed=True, mimetype=part.mime_type))
                break


### El atajo `.output_video`

Para simplificar el día a día, el SDK `google-genai` ofrece la propiedad `.output_video`, que siempre contiene el último video generado:

In [ ]:
# Acceder directamente al video generado sin inspeccionar los steps
video_part = interaction.output_video
print(video_part.mime_type)  # p.ej. 'video/mp4'
display(Video(data=video_part.data, embed=True, mimetype=video_part.mime_type))

<a name="helpers"></a>
## 4. Funciones auxiliares para mostrar videos

Para simplificar el resto del notebook, definimos funciones de ayuda basadas en `IPython.display.Video` y `Markdown`:

In [ ]:
# @title Funciones de ayuda para mostrar contenido
import base64
from IPython.display import Image, Markdown, Video, display


def get_output_video(interaction):
    """Retrieve the video part using .output_video or manual traversal."""
    if hasattr(interaction, "output_video") and interaction.output_video:
        return interaction.output_video
    if hasattr(interaction, "steps") and interaction.steps:
        for step in reversed(interaction.steps):
            if step.type == "model_output" and step.content:
                for content in reversed(step.content):
                    if content.type == "video":
                        return content
    return None



def get_output_image(interaction):
    """Obtiene la parte de imagen usando .output_image o recorriendo los steps manualmente."""
    if hasattr(interaction, "output_image") and interaction.output_image:
        return interaction.output_image
    if hasattr(interaction, "steps") and interaction.steps:
        for step in reversed(interaction.steps):
            if step.type != "model_output":
                continue
            for part in reversed(getattr(step, "content", None) or []):
                if part.type == "image":
                    return part
    return None


def show_video(interaction, width=640):
    """Muestra el video de salida de una interacción."""
    part = get_output_video(interaction)
    if part is None:
        print("⚠️ No se encontró ningún video en esta interacción.")
        return
    display(Video(data=part.data, embed=True, mimetype=part.mime_type, width=width))


def show_image(interaction, width=480):
    """Muestra la imagen de salida de una interacción."""
    part = get_output_image(interaction)
    if part is None:
        print("⚠️ No se encontró ninguna imagen en esta interacción.")
        return
    # part.data llega codificado en base64 (str); IPython.display.Image espera bytes crudos
    image_bytes = base64.b64decode(part.data) if isinstance(part.data, str) else part.data
    display(Image(data=image_bytes, width=width))


def show_thoughts(interaction):
    """Muestra los resúmenes de razonamiento (thinking) si existen."""
    # NOTA: los steps de tipo "thought" (ThoughtStep) no siempre exponen 'content';
    # por eso usamos getattr con valores por defecto en lugar de acceder directo al atributo.
    for step in interaction.steps:
        if step.type != "thought":
            continue
        thought_parts = getattr(step, "content", None) or getattr(step, "summaries", None) or []
        for part in thought_parts:
            text = getattr(part, "text", None) or (part if isinstance(part, str) else None)
            if text:
                display(Markdown(f"> 🧠 **Pensamiento del modelo:** {text}"))


<a name="thinking"></a>
## 5. Controlando el "pensamiento" del modelo

Gemini Omni Flash incorpora un razonamiento interno para planificar física, iluminación, interacción de objetos y el timing cinematográfico antes de sintetizar los frames. Puedes controlar este comportamiento dentro de `generation_config`:

* **Nivel de pensamiento**: usa `thinking_level` como `'low'` o `'high'` según la complejidad de la tarea. `'low'` reduce el razonamiento profundo (aunque sigue habiendo una planificación mínima).
* **Resúmenes**: con `thinking_summaries: "auto"` puedes pedir que el modelo te explique en texto qué estuvo planificando.

Ejemplo con un tema que se beneficia de mucho razonamiento: una explicación científica en stop-motion de plastilina.

In [ ]:
thinking_prompt = """
    Explicación tipo claymation de cómo funciona un transformer en AI, todo hecho
    de plastilina, sin manos, stop motion, científicamente preciso.
"""

interaction_thinking = client.interactions.create(
    model=MODEL_ID,
    input=thinking_prompt,
    generation_config={
        "thinking_level": "high",
        "thinking_summaries": "auto",
    },
)

show_thoughts(interaction_thinking)
show_video(interaction_thinking)

<a name="aspect_ratio"></a>
## 6. Relación de aspecto: horizontal y vertical

Por defecto, Omni genera videos en formato panorámico 16:9. Puedes generar video vertical para redes sociales (Reels, Shorts, TikTok) fijando `aspect_ratio` a `"9:16"` dentro de `response_format`:

In [ ]:
portrait_prompt = """
    Un smartphone flota inmóvil en el centro del encuadre sobre un fondo de estudio con
    degradado de morado a azul. En su pantalla se reproduce, en tiempo real, la misma
    escena que se está grabando en ese instante, creando un efecto de pantalla-dentro-de-
    pantalla que se repite hacia el infinito. Iluminación suave de producto, formato
    vertical pensado para redes sociales. Sin diálogo.
"""

interaction_portrait = client.interactions.create(
    model=MODEL_ID,
    input=portrait_prompt,
    response_format={
        "type": "video",
        "aspect_ratio": "9:16",
    },
)

show_video(interaction_portrait, width=360)


<a name="resolucion"></a>
## 7. Resolución: de borrador rápido a 4K

Ajusta el parámetro `resolution` dentro de `response_format` para controlar calidad y velocidad:

* **`360p` (modo borrador)**: clips ligeros de previsualización, roughly 2x más rápidos y con menor consumo de tokens. Ideal para iterar rápido sobre una idea antes de gastar tiempo en la versión final.
* **Alta resolución (`1080p`, `4k`)**: videos nítidos, listos para entrega final.

In [ ]:
# Generamos un video en alta resolución (1080p) con entrega por URI
interaction_360p = client.interactions.create(
    model=MODEL_ID,
    input="Un plano de drone cinematográfico deslizándose suavemente entre montañas de pinos con niebla al amanecer.",
    response_format={
        "type": "video",
        "resolution": "360p",  # Valores soportados: "360p", "720p", "1080p", "4k"
        "delivery": "uri"
    }
)

print(f"Video 1080p solicitado: {interaction_360p.id}")

In [ ]:
# 2. Extract file name and poll for ACTIVE state
video_output = interaction_360p.output_video
video_uri = video_output.uri
video_bytes = client.files.download(file=video_uri)
with open("mountain_sunrise.mp4", "wb") as f:
    f.write(video_bytes)

print(f"Downloaded mountain_sunrise.mp4 successfully from {video_uri}.")
display(Video("mountain_sunrise.mp4", embed=True))

<a name="world_knowledge"></a>
## 8. Ejemplo creativo: conocimiento del mundo

Uno de los puntos fuertes de Omni es su conocimiento profundo del mundo real, aplicado con precisión "científica" incluso en escenas complejas de física y astrofísica. Aquí generamos un video tipo documental sobre un agujero negro deformando la luz a su alrededor, con texto en pantalla correctamente renderizado:

In [ ]:
world_knowledge_prompt = """
    Recreación científicamente precisa de un agujero negro deformando la luz a su
    alrededor mediante lente gravitacional: un disco de acreción brillante gira a gran
    velocidad mientras las estrellas del fondo se curvan visiblemente cerca del horizonte
    de sucesos. Aparece un rótulo flotante que dice "Horizonte de sucesos" señalando el
    borde oscuro. Estilo documental cinematográfico tipo IMAX, música orquestal ambiental
    de fondo, sin diálogo.
"""

interaction_wk = client.interactions.create(
    model=MODEL_ID,
    input=world_knowledge_prompt,
)

show_video(interaction_wk)


<a name="image_to_video"></a>
## 9. De imagen a video

### Generando una imagen de referencia

En lugar de depender de archivos externos, puedes usar el modelo de generación de imágenes de Gemini (**Nano-Banana 2 Lite**, vía `gemini-3.1-flash-lite-image`) para crear rápidamente artwork de referencia en 16:9:

In [ ]:
prompt = "Crea una ilustración 3D estilo Pixar de un simpático robot reportero con una pantalla en el pecho que muestra el texto \'EN VIVO\', sosteniendo un micrófono con un ícono de rayo, sobre un fondo de estudio degradado de azul a morado." # @param {type:"string"}

# Genera una imagen de referencia personalizada en 16:9 usando Nano-Banana
interaction_img = client.interactions.create(
    model="gemini-3.1-flash-lite-image",
    input=prompt,
    response_modalities=["image"],
    generation_config={"image_config": {"aspect_ratio": "16:9"}},
)

image_part = get_output_image(interaction_img)
show_image(interaction_img)


### Imagen a video (frame inicial)

Para crear un video que empiece exactamente con la imagen generada, pasa tanto la imagen como tus instrucciones de animación dentro de `input`.

Fíjate en que `input` se estructura como una lista de diccionarios de distintos tipos (`{"type": "image", ...}` junto a `{"type": "text", ...}`). Esta lista multimodal te permite combinar una composición visual inicial explícita con instrucciones de animación en lenguaje natural, todo en la misma llamada:

In [ ]:
animate_prompt = "Anima al robot reportero encendiendo la pantalla de su pecho, donde parpadea en rojo la palabra \'EN VIVO\'; levanta el micrófono y guiña un ojo a cámara mientras pequeñas chispas de datos azules flotan a su alrededor." # @param {type:"string"}

interaction_i2v = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "image", "data": image_part.data, "mime_type": image_part.mime_type},
        {
            "type": "text",
            "text": animate_prompt,
        },
    ],
)

show_video(interaction_i2v)


<a name="reference_to_video"></a>
## 10. Video a partir de imágenes de referencia

En los flujos de referencia-a-video, la imagen proporcionada **no** se usa como frame literal inicial. En su lugar, el modelo extrae el sujeto, la identidad del personaje o la composición artística, y la incorpora en un entorno o perspectiva de cámara completamente distinto.

En este ejemplo, observa cómo nuestro robot reportero ilustrado por Nano-Banana se extrae como sujeto de referencia y aparece presentando las noticias en uno de los monitores de una sala de control de transmisión en vivo:


In [ ]:
ref_prompt = "Un plano en mano continuo recorriendo lentamente una sala de control de transmisión en vivo llena de monitores parpadeantes. En uno de los monitores centrales aparece presentando las noticias del día el robot exacto de la imagen de referencia." # @param {type:"string"}

interaction_ref = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "image", "data": image_part.data, "mime_type": image_part.mime_type},
        {"type": "text", "text": ref_prompt},
    ],
    generation_config={"video_config": {"task": "reference_to_video"}},
)

show_video(interaction_ref)


<a name="keyframes"></a>
## 11. Interpolación de keyframes

Genera un video continuo que hace una transición suave entre un frame inicial y un frame final. Ideal para órbitas de cámara, efectos de zoom, timelapses y loops perfectos.

In [ ]:
# Generamos dos imágenes distintas para interpolar entre ellas: una oruga y una mariposa
img_start = client.interactions.create(model="gemini-3.1-flash-lite-image", input="Ilustración realista de una oruga verde apoyada sobre una hoja, fondo de bosque suavemente desenfocado.", response_modalities=["image"])
img_end = client.interactions.create(model="gemini-3.1-flash-lite-image", input="Reemplaza la oruga con una mariposa monarca, conserva el resto de la imagen a la perfeccion", response_modalities=["text", "image"], previous_interaction_id=img_start.id)

frame_start = get_output_image(img_start)
frame_end = get_output_image(img_end)

In [ ]:
show_image(img_start)
show_image(img_end)

In [ ]:
# Interpolación de keyframes con frame inicial y final
# El parámetro `task` es opcional; si se omite, el modelo infiere automáticamente el modo.
interaction_interp = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "image", "data": frame_start.data, "mime_type": frame_start.mime_type},
        {"type": "image", "data": frame_end.data, "mime_type": frame_end.mime_type},
        {"type": "text", "text": "Transformación fluida de metamorfosis: alrededor de la oruga se teje un capullo que después se abre lentamente, revelando a la mariposa. Iluminación cálida de bosque."},
    ],
)

show_video(interaction_interp)


<a name="task_param"></a>
## 12. Control explícito con el parámetro `task`

Cuando se le dan imágenes y texto a la vez, Omni Flash infiere si debe usar una imagen como frame de apertura literal (`image_to_video`) o como una referencia indirecta de estilo/sujeto (`reference_to_video`). En prompts multimodales ambiguos, el modelo puede confundir una imagen de referencia con un frame inicial literal.

Para garantizar un comportamiento preciso en producción, fija explícitamente el parámetro `task` dentro de `generation_config.video_config`. Compara el mismo prompt con ambos modos:

In [ ]:
# Ejemplo con task="image_to_video": la imagen es el frame literal inicial
task_prompt = "Una sala de servidores futurista a oscuras se ilumina de golpe con luces de alerta roja parpadeantes. ¿Quién está manipulando la consola principal?" # @param {type:"string"}
interaction_task_i2v = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "image", "data": image_part.data, "mime_type": image_part.mime_type},
        {"type": "text", "text": task_prompt},
    ],
    generation_config={"video_config": {"task": "image_to_video"}},
)

show_video(interaction_task_i2v)


In [ ]:
# Mismo prompt, pero con task="reference_to_video": la imagen es solo referencia de estilo/sujeto
interaction_task_ref = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "image", "data": image_part.data, "mime_type": image_part.mime_type},
        {"type": "text", "text": task_prompt},
    ],
    generation_config={"video_config": {"task": "reference_to_video"}},
)

show_video(interaction_task_ref)


<a name="multi_ref"></a>
## 13. Escenas con múltiples personajes y referencias

Para escenas complejas con varios sujetos, usa etiquetas de referencia explícitas dentro de tu prompt de texto para asociar imágenes concretas a roles distintos:

* `<FIRST_FRAME>`: fija la imagen como el frame exacto de apertura del clip.
* `<IMAGE_REF_0>`, `<IMAGE_REF_1>`, etc.: asocia imágenes (indexadas desde 0, en el mismo orden que en `input`) a identidades de personajes, objetos o prendas específicas.

Omni Flash soporta combinar hasta 5 imágenes de referencia en una sola generación. Vamos a crear una escena tipo "celebración de hito de suscriptores" con tres mascotas y tres accesorios de creador de contenido:


In [ ]:
# Generamos 3 mascotas distintas y 3 accesorios de creador de contenido usando Nano-Banana
char1 = client.interactions.create(model="gemini-3.1-flash-lite-image", input="Crea una ilustración digital detallada de un zorro naranja con actitud desenfadada, sentado, sobre un fondo blanco liso.", response_modalities=["image"])
char2 = client.interactions.create(model="gemini-3.1-flash-lite-image", input="Crea una ilustración digital detallada de un panda rojo curioso, sentado, sobre un fondo blanco liso.", response_modalities=["image"])
char3 = client.interactions.create(model="gemini-3.1-flash-lite-image", input="Crea una ilustración digital detallada de un mapache travieso con gafas redondas, sentado, sobre un fondo blanco liso.", response_modalities=["image"])

acc1 = client.interactions.create(model="gemini-3.1-flash-lite-image", input="Crea una ilustración detallada de unos audífonos gamer con luces neón moradas encendidas, sobre un fondo blanco liso.", response_modalities=["image"])
acc2 = client.interactions.create(model="gemini-3.1-flash-lite-image", input="Crea una ilustración detallada de una gorra snapback negra con un ícono de botón de reproducción (play) bordado en el frente, sobre un fondo blanco liso.", response_modalities=["image"])
acc3 = client.interactions.create(model="gemini-3.1-flash-lite-image", input="Crea una ilustración detallada de un trofeo dorado brillante con forma de botón de Play, sobre un fondo blanco liso.", response_modalities=["image"])

img_c1, img_c2, img_c3 = get_output_image(char1), get_output_image(char2), get_output_image(char3)
img_a1, img_a2, img_a3 = get_output_image(acc1), get_output_image(acc2), get_output_image(acc3)


In [ ]:
show_image(char1)
show_image(char2)
show_image(char3)

In [ ]:
show_image(acc1)
show_image(acc2)
show_image(acc3)

Combina las seis referencias en una única escena de celebración usando las etiquetas `<IMAGE_REF_0>` a `<IMAGE_REF_5>`:


In [ ]:
multi_char_prompt = """
    En una única escena continua dentro de un set de streaming iluminado con neones:
    <IMAGE_REF_0> lleva puesto <IMAGE_REF_3>,
    <IMAGE_REF_1> lleva puesto <IMAGE_REF_4>,
    y <IMAGE_REF_2> sostiene con orgullo <IMAGE_REF_5>.
    Celebran juntos un nuevo hito de suscriptores mientras cae confeti dorado desde arriba.
"""

interaction_6refs = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "image", "data": img_c1.data, "mime_type": img_c1.mime_type},
        {"type": "image", "data": img_c2.data, "mime_type": img_c2.mime_type},
        {"type": "image", "data": img_c3.data, "mime_type": img_c3.mime_type},
        {"type": "image", "data": img_a1.data, "mime_type": img_a1.mime_type},
        {"type": "image", "data": img_a2.data, "mime_type": img_a2.mime_type},
        {"type": "image", "data": img_a3.data, "mime_type": img_a3.mime_type},
        {"type": "text", "text": multi_char_prompt},
    ],
)

show_video(interaction_6refs)


Fíjate en cómo Omni Flash asocia con precisión cada par de imagen-accesorio a su personaje correspondiente (los audífonos, la gorra y el trofeo terminan exactamente donde se pidió), sin mezclar identidades entre sí.


<a name="extension"></a>
## 14. Extender un video

Extiende un video existente generando una continuación fluida al final del clip. Describe en tu prompt cómo quieres que continúe, por ejemplo `"Continúa el video"` o `"Continúa la escena: la cámara recorre las montañas"`. Gemini Omni Flash analiza hasta 10 segundos de contexto previo para generar una continuación natural de 3 a 10 segundos, manteniendo continuidad temporal, identidad de personajes, inercia de cámara e iluminación ambiental.

### Ejemplo: extensión multi-turno (cámara y continuación de escena)

Primero generamos un clip base. Luego extendemos la escena a través de turnos de conversación, pasando `previous_interaction_id`:

In [ ]:
# Generamos el video base (Turno 1)
turn1_prompt = "[0-10s] Toma aérea continua e ininterrumpida siguiendo el ascenso de un cohete plateado que despega de una plataforma de lanzamiento entre nubes de humo blanco, la cámara se eleva junto al cohete. Toma completamente continua, sin cortes." # @param {type:"string"}
turn1 = client.interactions.create(
    model=MODEL_ID,
    input=turn1_prompt,
    response_format={"type": "video"},
)

print("Video base (Turno 1):")
show_video(turn1)


In [ ]:
# Extendemos el plano del Turno 1 usando extensión basada en prompt
# El modelo analiza la interacción previa y añade una continuación de forma fluida
extended_turn2 = client.interactions.create(
    model=MODEL_ID,
    previous_interaction_id=turn1.id,
    input="Continúa la toma: el cohete atraviesa las nubes hacia la atmósfera superior, revelando poco a poco la curvatura de la Tierra y un cielo lleno de estrellas.",
    response_format={"type": "video"},
)

print(f"ID del video extendido: {extended_turn2.id}")
show_video(extended_turn2)


<a name="edicion_multiturno"></a>
## 15. Edición conversacional multi-turno

Una de las capacidades más potentes de Omni es la edición conversacional de video en múltiples turnos. Usando el parámetro `previous_interaction_id`, puedes refinar iterativamente un clip generado a lo largo de varios turnos de conversación. El modelo mantiene el contexto de estado completo, aplicando modificaciones puntuales mientras preserva la continuidad visual.

Edita el video anterior describiendo únicamente los cambios que quieres hacer:

In [ ]:
# Editamos la generación anterior referenciando turn1.id
turn2_prompt = "Cambia el diseño del cohete a un esquema blanco y dorado con el logo de un canal de YouTube pintado en el costado. Deja todo lo demás igual." # @param {type:"string"}
turn2 = client.interactions.create(
    model=MODEL_ID,
    previous_interaction_id=turn1.id,
    input=turn2_prompt,
)

print("Video del Turno 2 (editado):")
show_video(turn2)


In [ ]:
# Otro turno de edición quirúrgica sobre el mismo video base
edit_prompt = "Añade una bandada de aves volando cerca de la plataforma de lanzamiento justo antes del despegue. Deja todo lo demás igual." # @param {type:"string"}
interaction_edit = client.interactions.create(
    model=MODEL_ID,
    previous_interaction_id=turn1.id,
    input=edit_prompt,
)
show_video(interaction_edit)


<a name="tu_video"></a>
## 16. 🎥 Sube TU video y edítalo

Hasta ahora hemos trabajado con videos generados por el propio modelo. Pero Omni Flash también permite **editar video real, grabado por ti** (por ejemplo, con tu celular), subiéndolo a través de la [Files API](https://ai.google.dev/gemini-api/docs/files).

> ⚠️ **Restricción regional importante**: subir videos propios para editarlos **no está disponible** en el Espacio Económico Europeo (EEE), Suiza, el Reino Unido y algunos estados de EE. UU. (Editar clips generados por IA, como en las secciones anteriores, sí funciona en todo el mundo). Si tu edición termina muy rápido y sin contenido de salida (`total_output_tokens: 0`), probablemente sea por esta restricción.
>
> 📏 **Duración máxima**: 10 segundos. Si tu video es más largo, recórtalo antes de subirlo (puedes usar el script `prep_video.py` del skill de Omni, o cualquier editor, para quedarte con el fragmento que quieras editar).

### Paso 1: sube tu archivo de video

Ejecuta la siguiente celda y selecciona un archivo `.mp4` desde tu computadora (máximo ~10 segundos, idealmente 720p/24fps para que la subida sea rápida).

In [ ]:
from google.colab import files

uploaded = files.upload()
my_video_path = next(iter(uploaded))  # nombre del primer archivo subido
print(f"Archivo subido: {my_video_path}")

### Paso 2: súbelo a la Files API

La Interactions API no acepta archivos locales directamente: primero hay que subirlos a través de `client.files.upload` y esperar a que su estado sea `ACTIVE`.

In [ ]:
import time

my_video_file = client.files.upload(file=my_video_path)

while my_video_file.state.name == "PROCESSING":
    print("Procesando video...")
    time.sleep(5)
    my_video_file = client.files.get(name=my_video_file.name)

print(f"Video listo: {my_video_file.uri} ({my_video_file.state.name})")

### Paso 3a: edítalo solo con un prompt de texto

La forma más simple: describe el cambio que quieres ver. Recuerda la regla de oro de la edición — **prompts cortos y quirúrgicos** funcionan mejor que descripciones largas.

Prueba, por ejemplo:
* `"Convierte este video en anime japonés. Mantén el sujeto principal sin cambios."`
* `"Cambia la iluminación a un atardecer dorado dramático."`
* `"Haz que empiece a nevar suavemente en la escena."`
* `"Quita el fondo y reemplázalo por una playa tropical."`

In [ ]:
my_edit_prompt = "Convierte el video al estilo de anime japonés." # @param {type:"string"}

interaction_my_video_edit = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "document", "uri": my_video_file.uri},
        {"type": "text", "text": my_edit_prompt},
    ],
)

show_video(interaction_my_video_edit)

### Paso 3b: edítalo con prompt + una imagen de referencia

Aquí es donde se pone realmente interesante: puedes combinar tu video con una **imagen de referencia** (un objeto, un personaje, una prenda, un logo...) para que Omni la incorpore dentro de tu video real.

Primero, sube o genera la imagen de referencia. Puedes:
1. Subir tu propia imagen (por ejemplo, el logo de tu marca, una prenda, un producto), o
2. Generarla con Nano-Banana como hicimos antes.

Aquí te dejamos las dos opciones — usa la que prefieras.

In [ ]:
# Opción A: sube tu propia imagen de referencia (por ejemplo, un producto, logo o prenda)
import base64
import mimetypes

uploaded_ref = files.upload()
my_ref_image_path = next(iter(uploaded_ref))

with open(my_ref_image_path, "rb") as f:
    my_ref_image_bytes = f.read()

my_ref_image_b64 = base64.b64encode(my_ref_image_bytes).decode("utf-8")
my_ref_mime_type = mimetypes.guess_type(my_ref_image_path)[0] or "image/png"

Ahora combina tu **video subido** (como fuente a editar) con la **imagen de referencia** (como objeto/estilo a incorporar) en una sola llamada. Usa la etiqueta `<IMAGE_REF_0>` en el prompt para dejar explícito el rol de la imagen:

In [ ]:
my_ref_edit_prompt = "Coloca <IMAGE_REF_0> puestas sobre la persona del video de forma natural, que combinen con el movimiento de la escena. Mantén todo lo demás igual." # @param {type:"string"}

interaction_my_video_ref_edit = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "document", "uri": my_video_file.uri},
        {"type": "image", "data": my_ref_image_b64, "mime_type": my_ref_mime_type},
        {"type": "text", "text": my_ref_edit_prompt},
    ],
)

show_video(interaction_my_video_ref_edit)

✅ **Resumen de esta sección**: para editar un video propio necesitas (1) subirlo con `client.files.upload` y esperar `ACTIVE`, (2) pasar `{"type": "document", "uri": video_file.uri}` dentro de `input`, y (3) opcionalmente añadir imágenes de referencia adicionales con `{"type": "image", ...}` antes de tu instrucción de texto.

<a name="uri_delivery"></a>
## 17. Entrega por URI para producción (`delivery="uri"`)

Al generar videos en entornos de producción, el beneficio clave de solicitar `delivery="uri"` dentro de `response_format` es la **fiabilidad de la conexión y la resiliencia de red**. Mientras que la entrega estándar (`inline`) transmite los datos en base64 directamente al cliente, una interrupción temporal de red o el refresco de una pestaña del navegador podría cortar el stream y perder el contenido generado.

Especificando `delivery="uri"`, la API aloja el video terminado de forma segura y te devuelve solo una referencia (URI) que puedes descargar cuando quieras, cuantas veces quieras:

In [ ]:
import time

# 1. Solicitamos el video con entrega por URI para mayor resiliencia de conexión
interaction_uri = client.interactions.create(
    model=MODEL_ID,
    input="Un amplio plano aéreo de drone sobre picos nevados de montaña al amanecer, en una única escena ininterrumpida.",
    response_format={
        "type": "video",
        "delivery": "uri"
    },
)

# 2. Extraemos el nombre del archivo y esperamos a que esté en estado ACTIVE
video_output = interaction_uri.output_video
video_uri = video_output.uri
video_bytes = client.files.download(file=video_uri)

with open("video_desde_uri.mp4", "wb") as f:
    f.write(video_bytes)

print("Video descargado en video_desde_uri.mp4")
display(Video("video_desde_uri.mp4", embed=True, width=640))

<a name="buenas_practicas"></a>
## 18. Guía de prompting: buenas prácticas

Para sacarle el máximo partido a Gemini Omni Flash, aplica estas técnicas de prompting estructurado:

### Consejo 1: continuidad de escena única
Por defecto, Omni puede introducir cortes de cámara multi-plano para darle dinamismo narrativo. Para forzar una perspectiva continua de una sola cámara, indícalo explícitamente:
* `"En una única escena ininterrumpida..."`
* `"Plano en mano continuo, sin cortes de escena..."`

In [ ]:
tip1_prompt = "Plano cenital fijo e ininterrumpido de un artista vertiendo pintura líquida de varios colores neón sobre un lienzo giratorio, formando lentamente un patrón fluido tipo marmolado (fluid art)." # @param {type:"string"}

interaction_tip1 = client.interactions.create(
    model=MODEL_ID,
    input=tip1_prompt,
)

show_video(interaction_tip1)


### Consejo 2: instrucciones negativas
Si aparecen elementos no deseados, indica prohibiciones claras en lenguaje natural:
* `"Sin diálogo ni narración."`
* `"Sin texto superpuesto en pantalla."`

In [ ]:
tip2_prompt = "Un rayo cae sobre una antigua torre de alta tensión durante una tormenta nocturna, chispas azules recorriendo los cables. Sin diálogo. Sin narración. Sin texto superpuesto en pantalla." # @param {type:"string"}

interaction_tip2 = client.interactions.create(
    model=MODEL_ID,
    input=tip2_prompt,
    response_format={"type": "video"},
)

show_video(interaction_tip2)


### Consejo 3: prompts de edición concisos
Al hacer ediciones multi-turno, mantén las instrucciones simples y quirúrgicas. Descripciones demasiado largas pueden confundir al motor de diferencias (diffing).

* **Sí:** `"Cambia el color del auto a azul metálico. Deja todo lo demás igual."`
* **No:** `"En el video que muestra el auto plateado circulando por la carretera, por favor reemplaza la pintura plateada por un tono azul metálico asegurándote de que..."`

### Consejo 4: control de tiempos y coreografía cronológica

Puedes pedir que ciertas acciones ocurran en momentos concretos del video, usando lenguaje natural o la sintaxis de timecode entre corchetes (`[0-3s]`, `[3-6s]`). Esto te da control total sobre el ritmo, el pacing y la narrativa por "beats":

* **Lenguaje natural**: `"después de 3 segundos, entra una mujer en escena"`, `"a los 5s empieza el estribillo en el audio de fondo"`.
* **Sintaxis de timecode**: `"[0-3s] Una persona camina..."`, `"[3-6s] Se detiene y se da la vuelta..."`.

In [ ]:
timing_prompt = "[0-3s] Una figura completamente cubierta de nieve permanece inmóvil en medio de un bosque congelado bajo la luz de la luna. [3-6s] La nieve se desprende de golpe, revelando que es una estatua de hielo tallada que lentamente abre los ojos." # @param {type:"string"}

interaction_timing = client.interactions.create(
    model=MODEL_ID,
    input=timing_prompt,
    response_format={"type": "video"},
)

show_video(interaction_timing)


### Consejo 5: comprensión de momentos clave

También puedes pedirle al modelo que edite el video cuando ocurra una acción específica, sin necesidad de especificar el timecode exacto tú mismo:

In [ ]:
tip5_turn1_prompt = "Un DJ mueve las manos sobre una mesa de mezclas en un escenario oscuro. A los 0:03 toca un botón y las luces del escenario se encienden en azul; a los 0:06 toca otro botón y cambian a rojo; a los 0:09 toca un tercer botón y estallan luces estroboscópicas multicolor." # @param {type:"string"}
tip5_turn1 = client.interactions.create(
    model=MODEL_ID,
    input=tip5_turn1_prompt,
)

print("Video del Turno 1 (base):")
show_video(tip5_turn1)


In [ ]:
tip5_turn2_prompt = "Cada vez que el DJ toca un botón, el género musical y el vestuario de la escena cambian por completo: electrónica, luego rock, luego salsa." # @param {type:"string"}
tip5_turn2 = client.interactions.create(
    model=MODEL_ID,
    input=tip5_turn2_prompt,
    previous_interaction_id=tip5_turn1.id,
)

print("Video del Turno 2 (editado):")
show_video(tip5_turn2)


### Consejo extra: el texto en pantalla funciona muy bien

A diferencia de modelos de video anteriores, el texto en los videos de Gemini Omni Flash se renderiza de forma correcta y legible. Puedes incluir cantidades razonables de texto y definir exactamente qué debe decir, incluso en elementos de fondo:

In [ ]:
text_prompt = """
    Una pantalla de smartphone en primer plano muestra una transmisión en vivo con la
    etiqueta "🔴 EN VIVO · 10,482 espectadores" en la esquina superior, y comentarios que
    van apareciendo en tiempo real desde abajo: "Esto es una locura 🔥", "¿Cómo hizo eso?",
    "Omni es el futuro". Fondo desenfocado de un creador de contenido sonriendo frente a
    un micrófono. Sin diálogo.
""" # @param {type:"string"}

interaction_text = client.interactions.create(
    model=MODEL_ID,
    input=text_prompt,
)

show_video(interaction_text)


<a name="veo_vs_omni"></a>
## 19. Veo vs. Omni: ¿cuál elegir?

Google ofrece dos grandes familias de modelos de generación de video: **Gemini Omni Flash** y **Veo**. Entender sus arquitecturas te ayuda a elegir la herramienta correcta para cada caso.

| Capacidad | Gemini Omni Flash (`gemini-omni-1.1-flash`) | Veo 3.1 (`veo-3.1-lite-generate-preview`) |
| :--- | :--- | :--- |
| **API principal** | **Interactions API** (`client.interactions.create`) | **Models API** (`client.models.generate_videos`) |
| **Fortaleza principal** | Edición conversacional multi-turno, iteración rápida, mezcla nativa de imagen+texto+video | Máxima fidelidad visual y control cinematográfico fino para producción final |
| **Casos de uso ideales** | Prototipado rápido, edición de video propio, contenido social iterativo, personajes/objetos consistentes vía referencias | Spots publicitarios, piezas cinematográficas de alta gama, control preciso de cámara |
| **Entrada de video real** | ✅ Sí (edición de video subido, sujeto a restricción regional) | Limitado |
| **Multi-turno con `previous_interaction_id`** | ✅ Nativo | No aplica de la misma forma |

**Regla práctica**: si necesitas iterar conversacionalmente, editar tu propio metraje o combinar varias referencias visuales, usa **Omni Flash**. Si necesitas la máxima calidad cinematográfica para una pieza final ya validada, considera **Veo**.

<a name="siguientes_pasos"></a>
## 20. Próximos pasos

* Explora la [documentación oficial de la Interactions API](https://ai.google.dev/gemini-api/docs/interactions-overview).
* Lee la [documentación de Gemini Omni](https://ai.google.dev/gemini-api/docs/omni) y la [documentación de Veo](https://ai.google.dev/gemini-api/docs/veo).
* Revisa la [tabla de precios de la Gemini API](https://ai.google.dev/gemini-api/docs/pricing) para los distintos niveles de generación de video.
* Consulta la [Introducción al diseño de prompts](https://ai.google.dev/gemini-api/docs/prompting-intro) para profundizar en técnicas de prompting.
* Si quieres automatizar estos flujos fuera de Colab (por ejemplo, en un pipeline local), puedes usar el skill `gemini-omni-flash-api`, que incluye scripts listos para generación, edición, y preprocesado de video con `ffmpeg` (recorte, verificación de duración/resolución, extracción/eliminación de audio).

¡Ahora ya tienes todo lo necesario para generar y editar video con Gemini Omni Flash — incluyendo tu propio contenido! 🎬